## Model 2: CatBoost

In [11]:
pip install catboost 


[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [12]:
import pandas as pd 
import numpy as np 
import catboost as cb 
from sklearn.model_selection import train_test_split, GridSearchCV, RandomizedSearchCV, cross_val_score, StratifiedKFold
from sklearn.metrics import accuracy_score, classification_report
import optuna

In [13]:
# importing data
irrigation = pd.read_csv('/Users/diyapatel/Documents/GitHub/Advanced-ML/Homework-2/train.csv')
irrigation.head(20)

,id,Soil_Type,Soil_pH,Soil_Moisture,Organic_Carbon,Electrical_Conductivity,Temperature_C,Humidity,Rainfall_mm,Sunlight_Hours,...,Crop_Type,Crop_Growth_Stage,Season,Irrigation_Type,Water_Source,Field_Area_hectare,Mulching_Used,Previous_Irrigation_mm,Region,Irrigation_Need
0,0,Loamy,4.92,32.58,1.01,3.05,15.01,50.61,725.99,5.90,...,Sugarcane,Sowing,Zaid,Drip,Rainwater,0.82,No,112.16,East,Low
1,1,Clay,7.08,56.61,0.44,2.00,22.92,67.86,985.66,6.98,...,Wheat,Vegetative,Kharif,Rainfed,River,5.27,Yes,47.16,South,Low
2,2,Clay,5.69,27.71,0.81,2.83,26.97,92.22,2201.70,6.05,...,Rice,Vegetative,Kharif,Sprinkler,Reservoir,8.24,Yes,110.38,North,Low
3,3,Sandy,5.65,13.32,1.33,0.87,13.32,61.57,1357.33,9.12,...,Wheat,Flowering,Kharif,Canal,River,8.32,Yes,53.85,South,Medium
4,4,Clay,7.96,59.14,0.38,0.96,20.22,91.11,1538.20,6.95,...,Wheat,Sowing,Rabi,Canal,River,7.37,No,93.19,South,Low
5,5,Sandy,5.09,24.70,1.28,0.48,12.21,92.35,696.03,9.11,...,Sugarcane,Flowering,Kharif,Sprinkler,River,0.81,No,86.56,South,Medium
6,6,Silt,7.53,49.67,1.44,1.62,14.02,61.65,889.39,8.44,...,Potato,Flowering,Zaid,Sprinkler,Rainwater,13.32,No,14.05,East,Medium
7,7,Loamy,7.56,48.61,0.38,1.31,22.78,61.53,708.82,9.96,...,Rice,Sowing,Zaid,Sprinkler,Reservoir,5.63,Yes,110.99,East,Low
8,8,Silt,6.02,53.01,0.90,0.49,20.55,61.30,1536.36,10.50,...,Potato,Flowering,Kharif,Rainfed,River,8.17,Yes,36.37,West,Low
9,9,Silt,7.39,41.91,0.58,0.78,39.25,85.52,281.76,8.86,...,Wheat,Vegetative,Kharif,Rainfed,Reservoir,5.41,No,71.92,Central,High


In [14]:
# splitting data into train and test sets
X = irrigation.drop(['Irrigation_Need', 'id'], axis=1)
y = irrigation['Irrigation_Need']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, shuffle=True, random_state=42) 


In [15]:
# Identify all columns in your training data that are of type 'object' (strings)
categorical_cols = X_train.select_dtypes(include=['object']).columns
# Convert those object columns to 'category' type in both train and test sets
for col in categorical_cols:
    X_train[col] = X_train[col].astype('category')
    X_test[col]  = X_test[col].astype('category')
print("Categorical columns converted to 'category' data type:")
print(list(categorical_cols))

Categorical columns converted to 'category' data type:
['Soil_Type', 'Crop_Type', 'Crop_Growth_Stage', 'Season', 'Irrigation_Type', 'Water_Source', 'Mulching_Used', 'Region']


In [16]:

cat_features = list(X_train.select_dtypes(include=['category', 'object']).columns)

# Initialize and train base model
cb_base = cb.CatBoostClassifier(
    random_state=42, 
    cat_features=cat_features,
    iterations=100,      # Drop the default from 1000 down to 100 so it's 10x faster
    thread_count=-1,    
    verbose=10           
)

cb_base.fit(X_train, y_train)

# Evaluate
y_pred_base = cb_base.predict(X_test)
print("\nBase Model Accuracy:", accuracy_score(y_test, y_pred_base))
print("\nBase Model Classification Report:\n", classification_report(y_test, y_pred_base))



Learning rate set to 0.5
0:	learn: 0.6384907	total: 342ms	remaining: 33.9s
10:	learn: 0.0668820	total: 2.95s	remaining: 23.8s
20:	learn: 0.0630590	total: 6.05s	remaining: 22.8s
30:	learn: 0.0618254	total: 8.71s	remaining: 19.4s
40:	learn: 0.0608397	total: 11.2s	remaining: 16.1s
50:	learn: 0.0602577	total: 13.9s	remaining: 13.3s
60:	learn: 0.0596617	total: 16.4s	remaining: 10.5s
70:	learn: 0.0591493	total: 19.2s	remaining: 7.83s
80:	learn: 0.0586804	total: 21.8s	remaining: 5.12s
90:	learn: 0.0583933	total: 24.6s	remaining: 2.43s
99:	learn: 0.0579357	total: 26.8s	remaining: 0us

Base Model Accuracy: 0.9850079365079365

Base Model Classification Report:
               precision    recall  f1-score   support

        High       0.96      0.92      0.94      4202
         Low       0.99      0.99      0.99     73983
      Medium       0.98      0.98      0.98     47815

    accuracy                           0.99    126000
   macro avg       0.98      0.96      0.97    126000
weighted avg  

## GridSearchCV Tuning

In [17]:
param_grid = {
    'iterations': [30, 50],         
    'learning_rate': [0.1, 0.2],         
    'depth': [3]                  
}
# 1. REMOVED cat_features from here
cb_grid = cb.CatBoostClassifier(
    random_state=42, 
    verbose=False,        
    thread_count=-1
)
grid_search = GridSearchCV(
    estimator=cb_grid,
    param_grid=param_grid,
    cv=2,                 
    scoring='accuracy',
    n_jobs=1,             
    verbose=2             
)
# 2. ADDED cat_features here instead. This safely bypasses the clone bug.
grid_search.fit(X_train, y_train, cat_features=cat_features)
y_pred_grid = grid_search.best_estimator_.predict(X_test)
print("\nBest GridSearchCV Parameters:", grid_search.best_params_)
print("GridSearchCV Accuracy:", accuracy_score(y_test, y_pred_grid))
print("\nGridSearchCV Classification Report:\n", classification_report(y_test, y_pred_grid))

Fitting 2 folds for each of 4 candidates, totalling 8 fits
[CV] END ..........depth=3, iterations=30, learning_rate=0.1; total time=   3.3s
[CV] END ..........depth=3, iterations=30, learning_rate=0.1; total time=   2.5s
[CV] END ..........depth=3, iterations=30, learning_rate=0.2; total time=   2.5s
[CV] END ..........depth=3, iterations=30, learning_rate=0.2; total time=   2.7s
[CV] END ..........depth=3, iterations=50, learning_rate=0.1; total time=   3.9s
[CV] END ..........depth=3, iterations=50, learning_rate=0.1; total time=   3.9s
[CV] END ..........depth=3, iterations=50, learning_rate=0.2; total time=   4.0s
[CV] END ..........depth=3, iterations=50, learning_rate=0.2; total time=   4.2s

Best GridSearchCV Parameters: {'depth': 3, 'iterations': 50, 'learning_rate': 0.2}
GridSearchCV Accuracy: 0.9844365079365079

GridSearchCV Classification Report:
               precision    recall  f1-score   support

        High       0.97      0.90      0.93      4202
         Low       0

## RandomizedSearchCV Tuning



In [18]:
param_dist = {
    'iterations': [30, 50, 70],
    'learning_rate': [0.1, 0.15, 0.2],
    'depth': [3, 4, 5]
}
# 1. REMOVED cat_features from here
cb_random = cb.CatBoostClassifier(
    random_state=42, 
    verbose=False, 
    thread_count=-1
)
random_search = RandomizedSearchCV(
    estimator=cb_random,
    param_distributions=param_dist,
    n_iter=5,      
    cv=2,                 
    scoring='accuracy',
    n_jobs=1,      
    random_state=42,
    verbose=1
)
# 2. ADDED cat_features here into the fit method instead
random_search.fit(X_train, y_train, cat_features=cat_features)
y_pred_random = random_search.best_estimator_.predict(X_test)
print("\nBest RandomizedSearchCV Parameters:", random_search.best_params_)
print("RandomizedSearchCV Classification Report:\n", classification_report(y_test, y_pred_random))

Fitting 2 folds for each of 5 candidates, totalling 10 fits

Best RandomizedSearchCV Parameters: {'learning_rate': 0.15, 'iterations': 50, 'depth': 4}
RandomizedSearchCV Classification Report:
               precision    recall  f1-score   support

        High       0.96      0.92      0.94      4202
         Low       0.99      0.99      0.99     73983
      Medium       0.98      0.98      0.98     47815

    accuracy                           0.98    126000
   macro avg       0.98      0.96      0.97    126000
weighted avg       0.98      0.98      0.98    126000



## Optuna Tuning


In [19]:


skf = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)

def cb_objective_fast(trial):
    iterations = trial.suggest_int('iterations', 30, 70)
    learning_rate = trial.suggest_float('learning_rate', 0.1, 0.2)
    depth = trial.suggest_int('depth', 3, 5)

    scores = []
    
    # FIX: We run the Cross Validation manually to completely bypass Scikit-Learn's clone bug!
    for train_idx, val_idx in skf.split(X_train, y_train):
        # Grab the specific data for this fold
        X_tr, y_tr = X_train.iloc[train_idx], y_train.iloc[train_idx]
        X_val, y_val = X_train.iloc[val_idx], y_train.iloc[val_idx]
        
        # We can safely pass cat_features natively here now since there's no wrapper to break!
        model = cb.CatBoostClassifier(
            random_state=42, 
            iterations=iterations, 
            learning_rate=learning_rate,  
            depth=depth, 
            cat_features=cat_features, # Put back safely
            verbose=False,
            thread_count=-1
        )
        
        model.fit(X_tr, y_tr)
        preds = model.predict(X_val)
        scores.append(accuracy_score(y_val, preds))
        
    return np.mean(scores)

study = optuna.create_study(direction='maximize')
optuna.logging.set_verbosity(optuna.logging.WARNING) 

study.optimize(cb_objective_fast, n_trials=5, show_progress_bar=True) 

print("\nBest Optuna Parameters:", study.best_params)

# Build best model with cat_features restored!
cb_optuna_best = cb.CatBoostClassifier(
    random_state=42, 
    iterations=study.best_params['iterations'], 
    learning_rate=study.best_params['learning_rate'], 
    depth=study.best_params['depth'],   
    cat_features=cat_features, # Put back safely
    verbose=False,
    thread_count=-1
)

cb_optuna_best.fit(X_train, y_train)

print("Optuna Classification Report:\n", classification_report(y_test, cb_optuna_best.predict(X_test)))



Best trial: 1. Best value: 0.98431: 100%|██████████| 5/5 [01:51<00:00, 22.23s/it] 



Best Optuna Parameters: {'iterations': 54, 'learning_rate': 0.1668376357298283, 'depth': 5}
Optuna Classification Report:
               precision    recall  f1-score   support

        High       0.96      0.92      0.94      4202
         Low       0.99      0.99      0.99     73983
      Medium       0.98      0.98      0.98     47815

    accuracy                           0.98    126000
   macro avg       0.98      0.96      0.97    126000
weighted avg       0.98      0.98      0.98    126000



In [20]:
test_df = pd.read_csv('/Users/diyapatel/Documents/GitHub/Advanced-ML/Homework-2/test.csv')

submission_ids = test_df['id']
X_test_final = test_df.drop(['id'], axis=1)

for col in categorical_cols:
    X_test_final[col] = X_test_final[col].astype('category')

final_predictions = cb_optuna_best.predict(X_test_final)

if final_predictions.ndim > 1:
    final_predictions = final_predictions.ravel()

submission = pd.DataFrame({
    'id': submission_ids,
    'Irrigation_Need': final_predictions
})

submission_path = '/Users/diyapatel/Documents/GitHub/Advanced-ML/Homework-2/catboost_submission.csv'
submission.to_csv(submission_path, index=False)